
# Optimizing Retention Targeting Strategy
### Predictive Modeling and Profit-Based Optimization

In [8]:
import warnings
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, roc_auc_score

warnings.filterwarnings('ignore')

In [9]:
X_train_resampled = pd.read_csv('../data/features/X_train_resampled.csv')
y_train_resampled = pd.read_csv('../data/features/y_train_resampled.csv').values.ravel()

X_test = pd.read_csv('../data/features/X_test.csv')
y_test = pd.read_csv('../data/features/y_test.csv').values.ravel()

### Part A: Predictive Modeling & Calibration

In [10]:
xgb_base = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_base.fit(X_train_resampled, y_train_resampled)

calibrated_xgb = CalibratedClassifierCV(xgb_base, method='isotonic', cv='prefit')
calibrated_xgb.fit(X_test, y_test)

calibrated_probs = calibrated_xgb.predict_proba(X_test)[:, 1]

print(f"Brier Score (Calibrated Accuracy): {brier_score_loss(y_test, calibrated_probs):.4f}")
print(f"Test ROC-AUC: {roc_auc_score(y_test, calibrated_probs):.4f}")

Brier Score (Calibrated Accuracy): 0.1732
Test ROC-AUC: 0.7799


### Part B: Profit-Based Optimization Framework
Now we take our accurate churn probabilities and simulate the retention campaign using the following business rules:
- **Campaign Cost ($c$):** $50 per targeted customer.
- **Intervention Success Rate ($\gamma$):** Assumed to be 20% (configurable).
- **Budget Constraint:** Max total spend of $200,000.
- **ROI Constraint:** Must achieve exactly $\ge$ 2.0 ROI (Incremental Revenue / Spend).


In [12]:

import joblib

portfolio = X_test.copy()

scaler = joblib.load('../data/features/scaler.pkl')

scaled_continuous = portfolio[['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV']]
unscaled_continuous = scaler.inverse_transform(scaled_continuous)

portfolio = pd.DataFrame()
portfolio['CLTV_Raw'] = unscaled_continuous[:, 3] 
portfolio['CLTV'] = portfolio['CLTV_Raw']

portfolio['True_Churn'] = y_test
portfolio['P_Churn'] = calibrated_probs

intervention_cost = 50.0  # $50 cost per offer
success_rate = 0.08       # Based on previous campaigns the success_rate is 8%

portfolio['Expected_Inc_Revenue'] = portfolio['P_Churn'] * success_rate * portfolio['CLTV']

portfolio['Expected_Inc_Profit'] = portfolio['Expected_Inc_Revenue'] - intervention_cost

portfolio = portfolio.sort_values(by='Expected_Inc_Profit', ascending=False).reset_index(drop=True)

profitable_candidates = portfolio[portfolio['Expected_Inc_Profit'] > 0]
print(f"Total theoretically profitable candidates (EIP > 0): {len(profitable_candidates)}")

Total theoretically profitable candidates (EIP > 0): 829


### Applying Budget & ROI Constraints

In [13]:

# Constraint 1: Budget Cap ($200k max)
max_customers_by_budget = int(200000 / intervention_cost) # 4,000 max

selected_portfolio = profitable_candidates.head(max_customers_by_budget)

# Constraint 2: Managerial ROI Threshold (ROI >= 2.0)
def calculate_roi(portfolio_subset):
    total_cost = len(portfolio_subset) * intervention_cost
    total_revenue = portfolio_subset['Expected_Inc_Revenue'].sum()
    return total_revenue / total_cost if total_cost > 0 else 0

current_roi = calculate_roi(selected_portfolio)

if len(selected_portfolio) > 0 and current_roi < 2.0:
    for i in range(len(selected_portfolio), 0, -1):
        subset = selected_portfolio.iloc[:i]
        if calculate_roi(subset) >= 2.0:
            selected_portfolio = subset
            break

final_roi = calculate_roi(selected_portfolio)
total_spend = len(selected_portfolio) * intervention_cost
total_expected_profit = selected_portfolio['Expected_Inc_Profit'].sum()

print("FINAL OPTIMIZED PORTFOLIO METRICS")
print("-"*30)
print(f"Customers Targeted: {len(selected_portfolio)}")
print(f"Total Budget Spent: ${total_spend:,.2f} / $200,000")
print(f"Projected ROI: {final_roi:.2f}x (Constraint: >= 2.0)")
print(f"Total Expected Profit: ${total_expected_profit:,.2f}")


FINAL OPTIMIZED PORTFOLIO METRICS
------------------------------
Customers Targeted: 829
Total Budget Spent: $41,450.00 / $200,000
Projected ROI: 2.84x (Constraint: >= 2.0)
Total Expected Profit: $76,440.87


### Strategy Comparison (Baseline vs Optimized) & Save Profitable Candidates

In [14]:

if len(selected_portfolio) > 0:
    baseline_portfolio = portfolio.sort_values(by='CLTV', ascending=False).head(len(selected_portfolio))
    baseline_roi = calculate_roi(baseline_portfolio)
    baseline_profit = baseline_portfolio['Expected_Inc_Profit'].sum()

    comparison_df = pd.DataFrame({
        'Strategy': ['Optimized (EIP Ranking)', 'Baseline (Top CLTV Ranking)'],
        'Customers Targeted': [len(selected_portfolio), len(baseline_portfolio)],
        'Total Spend': [f"${len(selected_portfolio) * intervention_cost:,.0f}", f"${len(baseline_portfolio) * intervention_cost:,.0f}"],
        'Expected Profit': [f"${total_expected_profit:,.0f}", f"${baseline_profit:,.0f}"],
        'ROI': [f"{final_roi:.2f}x", f"{baseline_roi:.2f}x"]
    })

    display(comparison_df)
else:
    print("No customers met the profitability threshold. Constraints prevented targeting any baseline users.")


,Strategy,Customers Targeted,Total Spend,Expected Profit,ROI
0,Optimized (EIP Ranking),829,"$41,450","$76,441",2.84x
1,Baseline (Top CLTV Ranking),829,"$41,450","$63,108",2.52x


In [18]:
import os
from sklearn.model_selection import train_test_split

# 1. Load the original dataset that still has the CustomerID column
original_df = pd.read_csv('../data/processed/telco_customer_churn_dataset_[VALIDATED].csv')

# 2. Recreate the EXACT row filtering from Phase 1 (Remove "Two year" contracts)
filtered_df = original_df[original_df['Contract'] != 'Two year'].copy()

# 3. Recreate the precise train/test split to perfectly isolate the 20% test set IDs
_, test_ids = train_test_split(
    filtered_df['CustomerID'],
    test_size=0.2, 
    random_state=42, 
    stratify=filtered_df['Churn Label']
)

# 4. Map the correct CustomerIDs safely back onto the full portfolio
portfolio['CustomerID'] = test_ids.values

# 5. Filter the portfolio down to ONLY the optimally selected individuals
final_export_df = portfolio.loc[selected_portfolio.index].copy()

# 6. Clean up the columns into a business-readable format
final_export_df = final_export_df[['CustomerID', 'CLTV', 'P_Churn', 'Expected_Inc_Profit']]
final_export_df = final_export_df.rename(columns={
    'CLTV': 'Customer_Lifetime_Value',
    'P_Churn': 'Calibrated_Churn_Probability',
    'Expected_Inc_Profit': 'Expected_ROI_Profit'
})

# 7. Save to the final CSV
os.makedirs('../data/exports', exist_ok=True)
export_path = '../data/exports/optimized_retention_audience_with_ids.csv'
final_export_df.to_csv(export_path, index=False)

print(f"Successfully exported {len(final_export_df)} profitable, target-ready candidates to: {export_path}")


Successfully exported 829 profitable, target-ready candidates to: ../data/exports/optimized_retention_audience_with_ids.csv
